# Análisis Cuantitativo y Evaluación de Riesgo del Portafolio
# Desarrollado por:  ** Equipo Dynasto**
### Trader & diveloper del portafolio: https://github.com/JuanTrader500
                      
---

**Propósito de este Notebook:**
Este documento contiene la validación estadística y el análisis de robustez de nuestro portafolio algorítmico multimercado (2021-2025). Su objetivo principal es someter las estrategias a pruebas de estrés probabilísticas mediante **Simulaciones de Montecarlo** para medir el Riesgo de Secuencia de Retornos (SRR), y analizar la **matriz de correlación** entre los sistemas para garantizar una diversificación estructural sólida y un control estricto del riesgo para nuestros inversores.

In [2]:
import pandas as pd 
import numpy as np 
import matplotlib.pyplot as plt
import os
import csv

In [3]:
'''
#Estrategias con 1% de riesgo por trade
strategys = [
    "C:/Users/juana/OneDrive/Escritorio//Backtest_Dynasto/BackTest_2021-2025/buy_on_friday_2021-2025.csv",
    "C:/Users/juana/OneDrive/Escritorio//Backtest_Dynasto/BackTest_2021-2025/dax_refactor_whit_tp20201-2025.csv",
    "C:/Users/juana/OneDrive/Escritorio//Backtest_Dynasto/BackTest_2021-2025/us30_50_100_notrailig_2021-2025.csv",
    "C:/Users/juana/OneDrive/Escritorio//Backtest_Dynasto/BackTest_2021-2025/us500_stable_strategy_2021-2025.csv",
    "C:/Users/juana/OneDrive/Escritorio/Backtest_Dynasto/BackTest_2021-2025/eurusd_induccion_2021-2025.csv",
    "C:/Users/juana/OneDrive/Escritorio//Backtest_Dynasto/BackTest_2021-2025/usdjpy_2021-2025.csv"
  
    ]
'''
strategys = [
    "C:/Users/juana/OneDrive/Escritorio/EquityBackTest/Backtest_Dynasto/BackTest_2021-2025/testus30_rr1_50.csv",
    "C:/Users/juana/OneDrive/Escritorio/EquityBackTest/Backtest_Dynasto/BackTest_2021-2025/testus30_rr2_50.csv"
  
    ]

In [18]:

#lista de archivos
strategys = [
    "C:/Users/juana/OneDrive/Escritorio//Backtest_Dynasto/BackTest_2021-2025/buy_on_friday_2021-2025.csv",
    "C:/Users/juana/OneDrive/Escritorio//Backtest_Dynasto/BackTest_2021-2025/dax_refactor_whit_tp20201-2025.csv",
    "C:/Users/juana/OneDrive/Escritorio//Backtest_Dynasto/BackTest_2021-2025/us30_50_100_notrailig_2021-2025.csv",
    "C:/Users/juana/OneDrive/Escritorio//Backtest_Dynasto/BackTest_2021-2025/us500_stable_strategy_2021-2025.csv",
    "C:/Users/juana/OneDrive/Escritorio/Backtest_Dynasto/BackTest_2021-2025/eurusd_induccion_2021-2025.csv",
    "C:/Users/juana/OneDrive/Escritorio//Backtest_Dynasto/BackTest_2021-2025/usdjpy_2021-2025.csv"
  
    ]

# Diccionario para guardar los DataFrames
# Clave = Nombre estrategia, Valor = DataFrame
mis_estrategias = {}

#los archivos que descargamos del backtesting son unpoco diferentes 
# asi que los tenemos que leer con utf-16
# Funcion auxiliar para encontrar donde empiezan los datos (la reutilizamos)
def encontrar_inicio_datos(ruta):
    try:
        with open(ruta, 'r', encoding='utf-16') as f:
            for i, line in enumerate(f):
                if "Hora de apertura" in line or "Open Time" in line:
                    return i
    except:
        # Intento con utf-8 si falla utf-16
        with open(ruta, 'r', encoding='utf-8') as f:
            for i, line in enumerate(f):
                if "Hora de apertura" in line or "Open Time" in line:
                    return i
    return 0

#  Bucle para procesar cada archivo
for archivo in strategys:
    # Obtener el nombre limpio del archivo (ej: "breakoutger40")
    nombre_estrategia = os.path.basename(archivo).replace('.csv', '')
    
    print(f"Procesando: {nombre_estrategia}...")
    
    # Encontrar fila de encabezado
    fila_header = encontrar_inicio_datos(archivo)
    
    try:
        # Leer con Pandas (usando la logica de limpieza)
        df = pd.read_csv(archivo, 
                         skiprows=fila_header, 
                         encoding='utf-16', 
                         sep='\t') # O sep=None engine='python'
        
        # Limpieza báaica
        df = df.dropna(axis=1, how='all')
        
        # GUARDAR EN EL DICCIONARIO
        mis_estrategias[nombre_estrategia] = df
        
    except Exception as e:
        print(f"Error leyendo {nombre_estrategia}: {e}")

print("\nCarga completa... Estrategias disponibles:")
print(mis_estrategias.keys())



Procesando: buy_on_friday_2021-2025...
Procesando: dax_refactor_whit_tp20201-2025...
Procesando: us30_50_100_notrailig_2021-2025...
Procesando: us500_stable_strategy_2021-2025...
Procesando: eurusd_induccion_2021-2025...
Procesando: usdjpy_2021-2025...

Carga completa... Estrategias disponibles:
dict_keys(['buy_on_friday_2021-2025', 'dax_refactor_whit_tp20201-2025', 'us30_50_100_notrailig_2021-2025', 'us500_stable_strategy_2021-2025', 'eurusd_induccion_2021-2025', 'usdjpy_2021-2025'])


# Graficas del comportamiento de las estrategias por separado

In [19]:
import plotly.graph_objects as go

# Creamos la figura interactiva vacía
fig = go.Figure()

print("Generando gráfico interactivo de Rendimiento y Drawdown...")

for nombre, df in mis_estrategias.items():
    # 1. Copia de seguridad
    data = df.copy()

    # 2. Convertir fecha (Usando tus columnas)
    data['<DATE>'] = pd.to_datetime(data['<DATE>'])
    
    # 3. Ordenar cronologicamente
    data = data.sort_values('<DATE>')
    
    # 4. Limpieza de Balance 
    if data['<BALANCE>'].dtype == 'O': 
        data['<BALANCE>'] = data['<BALANCE>'].astype(str).str.replace(' ', '').astype(float)
    
    # --- CALCULO DEL MAX DRAWDOWN ---
    # a. Calculamos el máximo histórico del balance hasta la fecha actual (Running Max)
    data['Peak_Balance'] = data['<BALANCE>'].cummax()
    
    # b. Calculamos el Drawdown porcentual en cada punto: (Balance - Pico) / Pico
    # Esto nos dará valores negativos (ej: -5.00 para una caída del 5%)
    data['Drawdown_Pct'] = ((data['<BALANCE>'] - data['Peak_Balance']) / data['Peak_Balance']) * 100
    
    # c. Obtenemos el PEOR Drawdown de toda la historia (el valor mínimo)
    max_dd = data['Drawdown_Pct'].min()

    # --- CÁLCULO DEL RENDIMIENTO PORCENTUAL ---
    balance_inicial = data['<BALANCE>'].iloc[0]
    
    if balance_inicial != 0:
        data['Performance_Pct'] = ((data['<BALANCE>'] / balance_inicial) - 1) * 10000
    else:
        data['Performance_Pct'] = 0

    # 5. Agregar la traza (línea) a la figura de Plotly
    # AÑADIMOS EL MAX DD AL NOMBRE DE LA LEYENDA
    etiqueta_leyenda = f"{nombre} (Max DD: {max_dd:.2f}%)"
    
    fig.add_trace(go.Scatter(
        x=data['<DATE>'],
        y=data['Performance_Pct'],
        mode='lines',
        name=etiqueta_leyenda,
        # Pasamos el drawdown como 'customdata' para mostrarlo en el tooltip
        customdata=data['Drawdown_Pct'],
        hovertemplate=(
            '<b>%{y:.2f}% Rendimiento</b><br>' +
            'Drawdown Actual: %{customdata:.2f}%<br>' +
            'Fecha: %{x|%Y-%m-%d}'
        )
    ))

# --- DISEÑO Y ESTÉTICA INTERACTIVA ---
fig.update_layout(
    title='Rendimiento Porcentual y Riesgo (Max Drawdown en Leyenda)',
    xaxis_title='Fecha',
    yaxis_title='Rendimiento (%)',
    template='plotly_dark', # Tema oscuro profesional
    hovermode="x unified",  # Muestra todas las etiquetas al mismo tiempo al pasar el mouse
    
    # --- TAMAÑO DE LA GRÁFICA ---
    width=1000,  
    height=600, 
    
    legend=dict(
        yanchor="top",
        y=0.99,
        xanchor="left",
        x=0.01,
        bgcolor="rgba(0,0,0,0.5)" # Fondo semitransparente para leer mejor
    )
)

# Mostrar gráfico
fig.show()

Generando gráfico interactivo de Rendimiento y Drawdown...


# Unificacion de las estrategias en un portafolio

In [13]:


# Creamos la figura interactiva vacía
fig = go.Figure()

print("Generando simulación de PORTAFOLIO UNIFICADO...")

# Lista para acumular TODAS las operaciones individuales de todas las estrategias
todas_las_operaciones = []

# --- 1. PROCESAMIENTO INDIVIDUAL Y EXTRACCIÓN DE TRADES ---
for nombre, df in mis_estrategias.items():
    # Copia y limpieza básica
    data = df.copy()
    data['<DATE>'] = pd.to_datetime(data['<DATE>'])
    data = data.sort_values('<DATE>')
    
    # Limpieza de Balance
    if data['<BALANCE>'].dtype == 'O': 
        data['<BALANCE>'] = data['<BALANCE>'].astype(str).str.replace(' ', '').astype(float)
    
    # --- INGENIERÍA INVERSA: OBTENER EL PROFIT POR TRADE ---
    # Asumimos que cada backtest empezo con 10,000 lo cual asi fue con los df que estamos trabajando 
    # El 'Profit' de cada trade es la diferencia entre el balance actual y el anterior.
    # Para la primera fila, el profit es (Balance - 10000).
    data['Trade_Profit'] = data['<BALANCE>'].diff()
    data.iloc[0, data.columns.get_loc('Trade_Profit')] = data.iloc[0]['<BALANCE>'] - 10000
    
    # Guardamos estas operaciones para el portafolio general
    temp_df = data[['<DATE>', 'Trade_Profit']].copy()
    temp_df['Estrategia'] = nombre # Etiquetamos de dónde vino
    todas_las_operaciones.append(temp_df)

    # --- CÁLCULO PARA LA LÍNEA INDIVIDUAL (Solo visualización) ---
    # Calculamos rendimiento porcentual individual para contexto
    balance_inicial = 10000 # Forzamos la base de 10k 
    data['Performance_Pct'] = ((data['<BALANCE>'] - balance_inicial) / balance_inicial) * 100
    
    # Max Drawdown Individual
    peak = data['<BALANCE>'].cummax()
    dd_pct = ((data['<BALANCE>'] - peak) / peak) * 100
    max_dd = dd_pct.min()

    # Graficamos la línea individual (más delgada y con transparencia)
    fig.add_trace(go.Scatter(
        x=data['<DATE>'],
        y=data['<BALANCE>'], # Graficamos DINERO, no porcentaje, para ver la simulación real
        mode='lines',
        name=f"{nombre} (DD: {max_dd:.2f}%)",
        opacity=0.5, # Un poco transparente para dar protagonismo al portafolio
        line=dict(width=1)
    ))

# --- 2. CONSTRUCCIÓN DEL PORTAFOLIO UNIFICADO ---
print("Fusionando operaciones y calculando curva combinada...")

# Unimos todas las listas en un solo DataFrame
df_portfolio = pd.concat(todas_las_operaciones)

# ORDENAMOS CRONOLÓGICAMENTE (Crucial para la simulación)
df_portfolio = df_portfolio.sort_values('<DATE>')

# --- SIMULACIÓN DE LA CUENTA ---
capital_inicial_portfolio = 10000

# La curva de equidad es el Capital Inicial + Suma Acumulada de todos los profits
df_portfolio['Equity_Portfolio'] = capital_inicial_portfolio + df_portfolio['Trade_Profit'].cumsum()

# --- CÁLCULO DE MÉTRICAS DEL PORTAFOLIO ---
# a. Max Drawdown del Portafolio
peak_port = df_portfolio['Equity_Portfolio'].cummax()
dd_port_series = ((df_portfolio['Equity_Portfolio'] - peak_port) / peak_port) * 100
max_dd_portfolio = dd_port_series.min()

# b. Retorno Total
retorno_total = df_portfolio['Equity_Portfolio'].iloc[-1] - capital_inicial_portfolio
retorno_pct = (retorno_total / capital_inicial_portfolio) * 100

# --- 3. GRAFICAR EL PORTAFOLIO MASTER ---
fig.add_trace(go.Scatter(
    x=df_portfolio['<DATE>'],
    y=df_portfolio['Equity_Portfolio'],
    mode='lines',
    name=f"PORTAFOLIO COMBINADO (DD: {max_dd_portfolio:.2f}%)",
    line=dict(color='white', width=2), # Línea blanca
    customdata=dd_port_series,
    hovertemplate=(
        '<b>$ %{y:,.2f} Balance</b><br>' +
        'Drawdown: %{customdata:.2f}%<br>' +
        'Estrategia origen: %{text}<br>' + 
        'Fecha: %{x|%Y-%m-%d}'
    )
))

# --- DISEÑO Y ESTÉTICA ---
fig.update_layout(
    title=f'Simulación de Portafolio Unificado (Capital Inicial: ${capital_inicial_portfolio:,.0f})<br><sup>Retorno Total: {retorno_pct:.2f}% | Max Drawdown Combinado: {max_dd_portfolio:.2f}%</sup>',
    xaxis_title='Fecha',
    yaxis_title='Balance de Cuenta ($)',
    template='plotly_dark',
    hovermode="x unified",
    width=1000,  
    height=600, 
    legend=dict(
        yanchor="top",
        y=0.99,
        xanchor="left",
        x=0.01,
        bgcolor="rgba(0,0,0,0.5)"
    )
)

fig.show()

Generando simulación de PORTAFOLIO UNIFICADO...
Fusionando operaciones y calculando curva combinada...


# Análisis de Matriz de Correlación: Decorrelación de Estrategias y Riesgo Sistémico

Este documento presenta el análisis de la matriz de correlación de retornos diarios entre los 5 algoritmos activos en el portafolio (buy_on_friday, dax_refactor, us30, us500, usdjpy) durante el periodo 2021-2025.

---

## Resumen Ejecutivo (Para Stakeholders e Inversores)

**¿Qué estamos viendo en esta matriz?**
La matriz de correlación mide qué tanto se mueven nuestras estrategias al mismo tiempo. Una puntuación de **1** significa que hacen exactamente lo mismo (ganan y pierden los mismos días), un **-1** significa que hacen lo opuesto, y un **0** significa que son completamente independientes. 

El resultado de nuestro portafolio es extraordinario: **Todas las estrategias tienen una correlación cercana a 0 (entre -0.01 y 0.06).**

**El Impacto en el Negocio (El "Santo Grial" de la Diversificación):**
En el mundo de las inversiones, el mayor peligro es tener estrategias que parezcan diferentes pero que pierdan dinero al mismo tiempo cuando el mercado cae. Esta matriz nos demuestra que hemos logrado la verdadera diversificación:
* **Independencia Real:** Un mal día para nuestra estrategia en el DAX no tiene ninguna relación con lo que hará nuestra estrategia en el USDJPY o el US500. 
* **Suavidad en las Ganancias:** Como las estrategias ganan y pierden en días distintos, actúan como un amortiguador natural entre ellas. Mientras una está en una racha de pérdidas, estadísticamente otra estará generando ganancias, estabilizando el balance general.
* **Reducción del Drawdown:** Esta es la razón principal por la cual nuestra la combinacion de las estrategias en un portafolio mostró una curva de capital (línea blanca) tan estable y con un Drawdown histórico de solo -11.78%(Ver siguiente analisis de Montecarlo para analisis de riesgo). 

**Conclusión:** No solo estamos operando diferentes activos; estamos operando *comportamientos* diferentes. El portafolio está estructuralmente protegido contra choques de mercado que afecten a un solo tipo de estrategia.Esto no quiere decir que no sea posible que algunos dias todas las estrategias salgan positivas o negativas pero si nos ayuda a darnos una idea de que tan probable es de que esto suceda.

---

## Análisis Cuantitativo (Technical Overview)

La matriz presenta los coeficientes de correlación de Pearson ($\rho$) para los retornos diarios logarítmicos de las 5 estrategias del portafolio. 

### Fundamento Matemático y Reducción de Varianza
De acuerdo con la Teoría Moderna de Portafolios (Markowitz), la varianza total de un portafolio ($\sigma_p^2$) no es simplemente el promedio de las varianzas individuales, sino que está dictada fuertemente por la covarianza entre sus componentes:

$$\sigma_p^2 = \sum_{i=1}^{n} w_i^2 \sigma_i^2 + \sum_{i=1}^{n} \sum_{j \neq i}^{n} w_i w_j \sigma_i \sigma_j \rho_{ij}$$

Dado que en nuestro portafolio el coeficiente de correlación cruzada es efectivamente nulo ($\rho_{ij} \approx 0$ para todo $i \neq j$), el segundo término de la ecuación (el componente de covarianza) se anula casi por completo. 

**Esto significa que el portafolio ha alcanzado el límite matemático máximo de reducción de riesgo a través de la diversificación pura (Ortogonalidad "Perpendicularidad entre los vectores").**

### Observaciones Cuantitativas Clave:

1. **Desacoplamiento de Beta (El caso US30 vs US500):** Históricamente, el índice Dow Jones (US30) y el S&P 500 (US500) tienen una correlación de mercado masiva. Sin embargo, la correlación entre nuestras estrategias sobre estos activos es de apenas **0.03**. Esto demuestra empíricamente que los algoritmos están capturando *Alpha* idiosincrático (ineficiencias específicas del modelo) y no simplemente acumulando *Beta* (exposición direccional al mercado de renta variable estadounidense).
2. **Ausencia de Correlación Negativa Falsa:** No hay valores cercanos a -1. Esto es positivo, ya que tener estrategias con alta correlación negativa a menudo significa que una estrategia está pagando los rendimientos de la otra (hedging destructivo de retornos). Al tener valores en $\sim 0$, obtenemos los beneficios de la diversificación sin sacrificar la Expectativa Matemática Positiva (EV) global del sistema.
3. **Robustez del Factor de Recuperación:** La independencia de los retornos (retornos ortogonales) justifica la resiliencia del portafolio observada en los tests de estrés(Ver la simulacion de montecarlo en la siguiente celda). La probabilidad de un evento de "cola gorda" (fat tail) simultáneo colapsa exponencialmente, reduciendo dramáticamente el Riesgo de Ruina sistémico acorde a la data historica.

**Recomendación Operativa:** El portafolio está listo para escalar capital. La adición de futuras estrategias debe pasar por este mismo filtro: cualquier nuevo algoritmo candidato debe demostrar un $\rho < 0.2$ con las estrategias existentes para ser admitido, garantizando que no se degrade la integridad estructural del portafolio actual.

In [14]:
# Diccionario para guardar los retornos diarios de cada estrategia
daily_returns_dict = {}

# Encontrar fecha de inicio y fin global para crear un eje temporal común
global_start = pd.Timestamp.max
global_end = pd.Timestamp.min

# Primera pasada: Detectar rango de fechas global
for nombre, df in mis_estrategias.items():
    dates = pd.to_datetime(df['<DATE>'])
    global_start = min(global_start, dates.min())
    global_end = max(global_end, dates.max())

# Crear un índice diario completo (incluyendo días sin operaciones)
# Usamos 'B' (Business Days) para excluir fines de semana si es Forex/Bolsa
all_days = pd.date_range(start=global_start, end=global_end, freq='D')

# Segunda pasada: Calcular curva de equidad diaria y retornos
for nombre, df in mis_estrategias.items():
    # Limpieza básica
    temp = df.copy()
    temp['<DATE>'] = pd.to_datetime(temp['<DATE>'])
    
    # Limpiar balance si es string
    if temp['<BALANCE>'].dtype == 'O':
        temp['<BALANCE>'] = temp['<BALANCE>'].astype(str).str.replace(' ', '').astype(float)
    
    # Nos quedamos con el ÚLTIMO balance de cada día (Cierre del día)
    temp = temp.set_index('<DATE>')
    daily_balance = temp['<BALANCE>'].resample('D').last()
    
    # Rellenar días vacíos (Forward Fill):
    # Si el martes no operó, su balance es igual al del lunes.
    daily_balance = daily_balance.reindex(all_days).ffill()
    
    # Rellenar el inicio (si empezó tarde) con el capital inicial (10,000)
    daily_balance = daily_balance.fillna(10000)
    
    # Calcular Retorno Diario Porcentual (Esto es lo que se correlaciona)
    # daily_returns = daily_balance.pct_change().fillna(0) 
    # Opcional: Usar diferencias logarítmicas para mayor precisión estadística

    
    daily_returns = np.log(daily_balance / daily_balance.shift(1)).fillna(0)
    
    daily_returns_dict[nombre] = daily_returns

# Crear DataFrame maestro de retornos
df_returns = pd.DataFrame(daily_returns_dict)

# --- 2. CÁLCULO DE LA MATRIZ DE CORRELACIÓN (PEARSON) ---
print("Calculando matriz de Pearson...")
correlation_matrix = df_returns.corr(method='pearson')

# Redondear 
correlation_matrix = correlation_matrix.round(2)

# --- 3. VISUALIZACIÓN: HEATMAP ---
print("Generando Heatmap...")

# Nombres de las estrategias para los ejes
names = correlation_matrix.columns.tolist()

fig = go.Figure(data=go.Heatmap(
    z=correlation_matrix.values,
    x=names,
    y=names,
    colorscale='RdBu_r', # Rojo = Positiva (Peligro), Azul = Negativa (Hedge)
    zmin=-1, 
    zmax=1,
    text=correlation_matrix.values, # Mostrar el número
    texttemplate="%{text}",         # Formato del texto
    textfont={"size": 14},
    hoverongaps=False
))

fig.update_layout(
    title='Matriz de Correlación de Retornos Diarios<br><sup>¿Que tan correlacionado esta mi retorno diario entre todas las estrategias?</sup>',
    xaxis_title="Estrategia",
    yaxis_title="Estrategia",
    width=800,
    height=800,
    template='plotly_dark',
    # Ajuste para que los ejes se vean bien
    xaxis=dict(side="bottom"),
    yaxis=dict(autorange="reversed") # Para que el índice 0 esté arriba
)

fig.show()

Calculando matriz de Pearson...
Generando Heatmap...


# Análisis de Robustez del Portafolio: Simulación de Montecarlo (N = 10,000)

Este documento presenta los resultados de la prueba de estrés probabilística aplicada al histórico de operaciones del portafolio, utilizando un modelo de Montecarlo de 10,000 iteraciones.

---

##  Resumen Ejecutivo (Para Stakeholders e Inversores)

**¿Qué estamos viendo en este gráfico?**
El éxito histórico de una estrategia de inversión siempre tiene un componente de suerte en el *orden* en que ocurren las ganancias y las pérdidas. Esta simulación responde a la pregunta vital: *"¿Qué pasaría si ejecutáramos exactamente las mismas operaciones, pero el mercado nos arrojara la peor racha de mala suerte posible?"*

Al mezclar el orden de las operaciones 10,000 veces, hemos creado 10,000 "vidas paralelas" para nuestro capital. Los resultados demuestran que la estrategia es **altamente robusta y rentable**:

* **La Realidad vs. El Riesgo (Línea Blanca vs. Línea Roja):** * Históricamente, tuvimos un viaje muy tranquilo con una caída máxima (Drawdown) de solo **-12.98%** (Línea Blanca). 
    * Sin embargo, el **Peor Caso Matemático** (Línea Roja) nos muestra que, si todas las operaciones perdedoras ocurrieran juntas al principio, el portafolio podría sufrir una caída del **-59.98%**.
* **Control del Riesgo de Ruina (0.1%):** A pesar de ese "peor caso" que asusta, la estadística nos dice que la probabilidad de que nuestro portafolio caiga un 50% es de apenas **1 en 1,000** (0.1%). 

**Conclusión de Negocio:** El sistema es confiable. Sin embargo, debemos ajustar nuestras expectativas emocionales: la caída histórica del 11% fue afortunada. En el futuro operativo, es estadísticamente normal y esperable enfrentar caídas (Drawdowns) que naveguen dentro de la zona azul oscura de probabilidad.

---

## Análisis Cuantitativo y Explicación Matemática (Technical Overview)

El propósito subyacente de este análisis es evaluar la vulnerabilidad del portafolio ante el **Riesgo de Secuencia de Retornos (Sequence of Returns Risk - SRR)** mediante un remuestreo no paramétrico (permutación aleatoria).

### Fundamento Matemático

Dado un conjunto histórico de retornos por operación $R = \{r_1, r_2, \dots, r_n\}$, una iteración de Montecarlo genera una permutación aleatoria $\pi(R)$. La curva de capital acumulado $E_t$ para una simulación específica en el paso $t$ se define como:

$$E_t = E_0 + \sum_{k=1}^{t} r_{\pi(k)}$$

*Donde $E_0$ es el capital inicial (\$10,000).*

Para medir el riesgo extremo, calculamos el *Drawdown* ($DD_t$) en cada punto de la serie de tiempo para cada una de las 10,000 simulaciones:

$$DD_t = \frac{E_t - \max_{0 \le k \le t}(E_k)}{\max_{0 \le k \le t}(E_k)}$$

El **Maximum Drawdown (Max DD)** reportado para el "Peor Caso" corresponde al límite inferior absoluto de esta distribución bidimensional: $\min(DD)$ a través de la matriz de simulaciones.

### Lectura Técnica de la Gráfica y Métricas

1. **Asimetría de la Curva Real (Fat Tails):** La curva de "REALIDAD" (línea blanca, Max DD: -12.98%) navega consistentemente en la frontera superior del intervalo de confianza del 90%. Esto indica que la secuencia histórica observada fue estadísticamente un *outlier* positivo (estuvo en el rango de los mejores escenarios posibles en cuanto a suavidad de la curva).
2. **Dispersión del Drawdown:** La inmensa divergencia entre el Max DD Real (-12.98%) y el Max DD del Peor Caso (-59.98%) sugiere la presencia de operaciones con *Heavy Tails* (pérdidas atípicas grandes). Cuando el muestreo aleatorio agrupa (clusteriza) estos eventos de cola en las primeras etapas de la simulación (visible en el profundo valle de la línea roja entre enero y julio de 2021), el capital sufre una erosión no lineal severa.
3. **Convergencia de la Nube de Probabilidad:** A pesar del riesgo de varianza a corto plazo, la "Zona de Probabilidad" (Banda del 5% al 95%) tiene una pendiente claramente positiva y un ancho controlado, lo que matemáticamente garantiza una alta tasa de recuperación (Recovery Factor) a largo plazo.

**Recomendación Cuantitativa:** Dado que el Riesgo de Ruina (definido como un DD > 50%) está aislado en la cola extrema inferior de la distribución gaussiana (0.1%), el modelo de *Position Sizing* actual es matemáticamente seguro. Sin embargo, el tamaño de las posiciones no debe aumentarse recomendada mente, ya que apalancar el portafolio lo expondria a cruzar el umbral del -50% con mayor frecuencia bajo escenarios aleatorios.

In [15]:
# Creamos la figura
fig = go.Figure()

print("1. Procesando datos históricos...")

# --- PROCESAMIENTO DE DATOS 
todas_las_operaciones = []
for nombre, df in mis_estrategias.items():
    data = df.copy()
    data['<DATE>'] = pd.to_datetime(data['<DATE>'])
    data = data.sort_values('<DATE>')
    
    if data['<BALANCE>'].dtype == 'O': 
        data['<BALANCE>'] = data['<BALANCE>'].astype(str).str.replace(' ', '').astype(float)
    
    data['Trade_Profit'] = data['<BALANCE>'].diff()
    data.iloc[0, data.columns.get_loc('Trade_Profit')] = data.iloc[0]['<BALANCE>'] - 10000
    
    temp_df = data[['<DATE>', 'Trade_Profit']].copy()
    todas_las_operaciones.append(temp_df)

df_portfolio = pd.concat(todas_las_operaciones)
df_portfolio = df_portfolio.sort_values('<DATE>')

capital_inicial_portfolio = 10000
# Calculamos la curva REAL para tener el eje X correcto
df_portfolio['Equity_Portfolio'] = capital_inicial_portfolio + df_portfolio['Trade_Profit'].cumsum()
fechas = df_portfolio['<DATE>'].values # Guardamos fechas para el eje X

# -----------------------------------------------------------------------------
# --- 2. MOTOR MONTECARLO DE ALTO RENDIMIENTO (VECTORIZADO) ---
# -----------------------------------------------------------------------------
NUM_SIMULACIONES = 10000  
print(f"2. Ejecutando {NUM_SIMULACIONES} simulaciones en memoria (NumPy)...")

#====== Seed de nustra simulacion para que los resuktados sena consistentes ========================
np.random.seed(69) # Locura Xd


profits_array = df_portfolio['Trade_Profit'].values
n_trades = len(profits_array)

# Matriz gigante para guardar TODOS los escenarios [10000 filas x N trades]
# Esto es mucho más rápido que usar listas
sim_matrix = np.zeros((NUM_SIMULACIONES, n_trades))

# Llenamos la matriz (La parte pesada de cálculo)
for i in range(NUM_SIMULACIONES):
    sim_matrix[i, :] = np.random.permutation(profits_array)

# Calculamos la Equidad Acumulada para toda la matriz de una vez
# axis=1 significa que sumamos a lo largo del tiempo (columnas)
equity_matrix = capital_inicial_portfolio + np.cumsum(sim_matrix, axis=1)

# --- CALCULO DE MÉTRICAS MASIVO ---
# 1. Calcular Max Drawdown para cada una de las 10,000 simulaciones
#    (Peak acumulado - Valor actual) / Peak acumulado
peaks = np.maximum.accumulate(equity_matrix, axis=1)
drawdowns = ((equity_matrix - peaks) / peaks) * 100 # CORREGIDO: * 100
max_dds = np.min(drawdowns, axis=1) # El mínimo (más negativo) de cada fila

# 2. Encontrar índices de los casos extremos
idx_peor_dd = np.argmin(max_dds)      # Índice de la simulación con el DD más profundo (MAE)
idx_mejor_mfe = np.argmax(equity_matrix[:, -1]) # Índice de la que terminó con más dinero (MFE)

# 3. Calcular la "Nube" estadística (Bandas de confianza 95%)
#    Esto evita graficar 10,000 líneas. Calculamos los bordes de la masa de datos.
lower_bound = np.percentile(equity_matrix, 5, axis=0)  # El 5% peor en cada punto
upper_bound = np.percentile(equity_matrix, 95, axis=0) # El 95% mejor en cada punto

print(f"   -> Peor Drawdown simulado: {max_dds[idx_peor_dd]:.2f}%")

#  EXTRACCION DE METRICAS CLAVE PARA EL NEGOCIO ---

# Definimos qué consideramos "Ruina" (Ej: Perder el 30% del capital máximo)
NIVEL_RUINA_PCT = -50.0 

# A. Probabilidad de Beneficio (¿Cuántas terminaron en verde?)
capitales_finales = equity_matrix[:, -1] # Tomamos solo la última columna (el final del periodo)
simulaciones_ganadoras = np.sum(capitales_finales > capital_inicial_portfolio)
probabilidad_beneficio = (simulaciones_ganadoras / NUM_SIMULACIONES) * 100

# B. Riesgo de Ruina (¿Cuántas tocaron nuestro límite inaceptable de pérdida?)
# Ya tenías calculado 'max_dds' (los peores drawdowns de cada fila)
simulaciones_quebradas = np.sum(max_dds <= NIVEL_RUINA_PCT)
riesgo_ruina = (simulaciones_quebradas / NUM_SIMULACIONES) * 100

# C. Retorno Mediano (La expectativa más realista, ignorando los extremos locos)
capital_mediano = np.median(capitales_finales)
retorno_mediano_pct = ((capital_mediano - capital_inicial_portfolio) / capital_inicial_portfolio) * 100

print(f"   -> Probabilidad de salir en positivo: {probabilidad_beneficio:.1f}%")
print(f"   -> Riesgo de Ruina (DD peor a {NIVEL_RUINA_PCT}%): {riesgo_ruina:.1f}%")

# -----------------------------------------------------------------------------
# --- 3. GRAFICACIÓN OPTIMIZADA (SOLO 4 TRAZOS + NUBE) ---
# -----------------------------------------------------------------------------
print("3. Generando gráfico optimizado con la nube azul y caso del backtest Vs Peor esenario...")

# A. LA NUBE (Banda de Confianza 5% - 95%)
# Truco: Graficamos el borde inferior invisible, y luego el superior rellenando hasta el inferior
fig.add_trace(go.Scatter(
    x=fechas, y=lower_bound,
    mode='lines', line=dict(width=0.5), 
    showlegend=False, hoverinfo='skip'
))
fig.add_trace(go.Scatter(
    x=fechas, y=upper_bound,
    mode='lines', line=dict(width=0.5), 
    fill='tonexty', # Rellenar hasta la traza anterior (lower_bound)
    fillcolor='rgba(0, 191, 255, 0.15)', # Azul cian muy suave
    name='Zona de Probabilidad (90%)',
    hoverinfo='skip'
))

# C. EL PEOR CASO (Max Drawdown)
fig.add_trace(go.Scatter(
    x=fechas, y=equity_matrix[idx_peor_dd],
    mode='lines', name=f'Peor Caso (DD: {max_dds[idx_peor_dd]:.2f}%)',
    line=dict(color='rgba(255, 50, 50, 0.8)', width=1.5) # Rojo solido
))

# D.PORTAFOLIO REAL (La Simulacion del 2021-2025)
# Recalculamos métricas reales para la leyenda
peak_real = df_portfolio['Equity_Portfolio'].cummax()
dd_real_series = ((df_portfolio['Equity_Portfolio'] - peak_real) / peak_real) * 100
max_dd_real = dd_real_series.min()

fig.add_trace(go.Scatter(
    x=fechas, y=df_portfolio['Equity_Portfolio'],
    mode='lines',
    name=f"REALIDAD (DD: {max_dd_real:.2f}%)",
    line=dict(color='white', width=3), # Blanco grueso
    customdata=dd_real_series,
    hovertemplate='<b>$%{y:,.0f}</b><br>DD: %{customdata:.2f}%'
))

# Diseño
fig.update_layout(
    title=f'Montecarlo 10,000 Simulaciones<br><sup>Realidad vs El Azar (Intervalo de Confianza 95%)</sup>',
    xaxis_title='Fecha', yaxis_title='Capital ($)',
    template='plotly_dark',
    hovermode="x unified",
    legend=dict(y=0.99, x=0.01, bgcolor="rgba(0,0,0,0.5)")
)

fig.show()

1. Procesando datos históricos...
2. Ejecutando 10000 simulaciones en memoria (NumPy)...
   -> Peor Drawdown simulado: -66.88%
   -> Probabilidad de salir en positivo: 100.0%
   -> Riesgo de Ruina (DD peor a -50.0%): 0.1%
3. Generando gráfico optimizado con la nube azul y caso del backtest Vs Peor esenario...


In [16]:
from plotly.subplots import make_subplots

print("Calculando métricas mensuales por año (Versión Estable)...")

# --- 1. PREPARACIÓN DE DATOS ---
# Añadimos la serie de Drawdown en porcentaje (tu variable original)
df_portfolio['DD_Pct'] = dd_port_series 

# CÁLCULO NUEVO: Drawdown en Dólares ($)
peak_port = df_portfolio['Equity_Portfolio'].cummax()
df_portfolio['DD_USD'] = df_portfolio['Equity_Portfolio'] - peak_port

# Aseguramos que la fecha es el índice para facilitar el remuestreo mensual
df_temp = df_portfolio.set_index('<DATE>')

# Agrupamos por mes ('ME') sumando beneficios y extrayendo los peores momentos (DD)
df_mensual = df_temp.resample('ME').agg(
    Profit_Mensual_USD=('Trade_Profit', 'sum'),
    Max_DD_Mensual_Pct=('DD_Pct', 'min'),
    Max_DD_Mensual_USD=('DD_USD', 'min'),
    Equity_Final=('Equity_Portfolio', 'last') # Necesario para sacar el % del mes
).reset_index()

# CÁLCULO NUEVO: Porcentaje Mensual Exacto
# Con cuánto dinero iniciamos el mes (es el dinero con el que finalizamos el anterior)
df_mensual['Equity_Inicial'] = df_mensual['Equity_Final'].shift(1)
df_mensual['Equity_Inicial'] = df_mensual['Equity_Inicial'].fillna(capital_inicial_portfolio)

# Limpiamos los meses donde no hubo ninguna operación ANTES de dividir para evitar errores
df_mensual = df_mensual.dropna(subset=['Profit_Mensual_USD'])

# Calculamos el % que representó el profit en base a lo que teníamos ese mes
df_mensual['Profit_Mensual_Pct'] = (df_mensual['Profit_Mensual_USD'] / df_mensual['Equity_Inicial']) * 100

# Extraemos el Año y damos formato al Mes para los ejes
df_mensual['Año'] = df_mensual['<DATE>'].dt.year
df_mensual['Mes_Format'] = df_mensual['<DATE>'].dt.strftime('%b') 

# --- 2. BUCLE PARA GRAFICAR AÑO POR AÑO ---
años_unicos = sorted(df_mensual['Año'].unique())

for año in años_unicos:
    # Filtramos la tabla para el año actual
    df_año = df_mensual[df_mensual['Año'] == año].copy()
    
    if df_año.empty:
        continue
        
    # Colores lógicos
    colores_barras = ['rgba(0, 200, 100, 0.7)' if p >= 0 else 'rgba(255, 50, 50, 0.7)' for p in df_año['Profit_Mensual_USD']]

    # Creamos la figura con doble eje Y
    fig_año = make_subplots(specs=[[{"secondary_y": True}]])

    # A. Histograma (Barras de Profit en USD y textos en %)
    fig_año.add_trace(
        go.Bar(
            x=df_año['Mes_Format'],
            y=df_año['Profit_Mensual_USD'],
            name="Resultado Mensual ($ y %)",
            marker_color=colores_barras,
            text=df_año['Profit_Mensual_Pct'].apply(lambda x: f"{x:+.2f}%"), # TEXTO FLOTANTE SOBRE BARRA
            textposition='outside',
            textfont=dict(color='white', size=11),
            customdata=df_año['Profit_Mensual_Pct'], # Pasamos 1 sola dimensión segura
            hovertemplate='<b>%{x} ' + str(año) + '</b><br>Profit: $ %{y:,.2f} (%{customdata:+.2f}%)<extra></extra>'
        ),
        secondary_y=False,
    )

    # B. Línea de Drawdown (en % y tooltips en USD)
    fig_año.add_trace(
        go.Scatter(
            x=df_año['Mes_Format'],
            y=df_año['Max_DD_Mensual_Pct'],
            name="Peor Momento (DD % y $)",
            mode='lines+markers',
            line=dict(color='orange', width=2),
            marker=dict(size=8, color='white', line=dict(width=2, color='orange')),
            customdata=df_año['Max_DD_Mensual_USD'], # Pasamos 1 sola dimensión segura
            hovertemplate='Estrés/DD Peor Momento: %{y:.2f}% ($ %{customdata:,.2f})<extra></extra>'
        ),
        secondary_y=True,
    )

    # --- DISEÑO DEL GRÁFICO DEL AÑO ---
    profit_total_año_usd = df_año['Profit_Mensual_USD'].sum()
    peor_dd_año_pct = df_año['Max_DD_Mensual_Pct'].min()
    
    fig_año.update_layout(
        title=f'<b>Desempeño Mensual y Estrés - {año}</b><br><sup>Profit Anual: ${profit_total_año_usd:,.2f} | Peor Caída del Año: {peor_dd_año_pct:.2f}%</sup>',
        template='plotly_dark',
        hovermode="x unified",
        width=950,
        height=450,
        showlegend=True,
        legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1),
        margin=dict(t=80) # Damos espacio superior para que el % fuera de la barra no se corte
    )

    # Ajuste dinámico del eje Y para las barras (evita que el texto de arriba se corte)
    max_y = df_año['Profit_Mensual_USD'].max()
    min_y = df_año['Profit_Mensual_USD'].min()
    margen_extra = max(abs(max_y), abs(min_y)) * 0.25 # Añade 25% de espacio
    
    fig_año.update_yaxes(title_text="<b>Beneficio ($)</b>", secondary_y=False, showgrid=False, range=[min_y - margen_extra, max_y + margen_extra])
    fig_año.update_yaxes(title_text="<b>Drawdown (%)</b>", secondary_y=True, showgrid=True, gridcolor='rgba(255,255,255,0.1)')

    # Imprimir la gráfica
    fig_año.show()

Calculando métricas mensuales por año (Versión Estable)...


In [17]:

import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.io as pio

pio.renderers.default = "notebook_connected"

capital_inicial_portfolio = 10000

# --- 1. PREPARACIÓN DE DATOS (MATEMÁTICA DE CAPITAL FIJO) ---
peak_port = df_portfolio['Equity_Portfolio'].cummax()
df_portfolio['DD_USD'] = df_portfolio['Equity_Portfolio'] - peak_port
df_portfolio['DD_Pct_Fijo'] = (df_portfolio['DD_USD'] / capital_inicial_portfolio) * 100

df_temp = df_portfolio.set_index('<DATE>')

df_mensual = df_temp.resample('ME').agg(
    Profit_Mensual_USD=('Trade_Profit', 'sum'),
    Max_DD_Mensual_Pct=('DD_Pct_Fijo', 'min'),
    Max_DD_Mensual_USD=('DD_USD', 'min')
).reset_index()

df_mensual = df_mensual.dropna()

df_mensual['Profit_Mensual_Pct'] = (df_mensual['Profit_Mensual_USD'] / capital_inicial_portfolio) * 100
df_mensual['Año'] = df_mensual['<DATE>'].dt.year
df_mensual['Mes_Format'] = df_mensual['<DATE>'].dt.strftime('%b')

df_mensual = df_mensual.replace([np.inf, -np.inf], np.nan).dropna()

años_unicos = sorted(df_mensual['Año'].unique())

# --- 2. BUCLE PARA GRAFICAR AÑO POR AÑO ---
for año in años_unicos:
    df_año = df_mensual[df_mensual['Año'] == año].copy()

    meses_list = [str(x) for x in df_año['Mes_Format'].tolist()]
    profit_usd_list = [float(x) for x in df_año['Profit_Mensual_USD'].tolist()]
    profit_pct_list = [float(x) for x in df_año['Profit_Mensual_Pct'].tolist()]
    dd_pct_list = [float(x) for x in df_año['Max_DD_Mensual_Pct'].tolist()]
    dd_usd_list = [float(x) for x in df_año['Max_DD_Mensual_USD'].tolist()]

    # ← NUEVO: Cálculo de medias
    media_profit_usd = sum(profit_usd_list) / len(profit_usd_list)
    media_profit_pct = sum(profit_pct_list) / len(profit_pct_list)
    media_dd_pct = sum(dd_pct_list) / len(dd_pct_list)

    colores_barras = ['rgba(0, 200, 100, 0.7)' if p >= 0 else 'rgba(255, 50, 50, 0.7)' for p in profit_usd_list]
    textos_fuera_barra = [f"{p:+.2f}%" for p in profit_pct_list]

    hover_barras = [f"<b>{m} {año}</b><br>Profit (Base 10K): $ {u:,.2f} ({p:+.2f}%)" for m, u, p in zip(meses_list, profit_usd_list, profit_pct_list)]
    hover_lineas = [f"Estrés/DD (Base 10K): {p:.2f}% ($ {u:,.2f})" for p, u in zip(dd_pct_list, dd_usd_list)]

    fig_año = make_subplots(specs=[[{"secondary_y": True}]])

    fig_año.add_trace(
        go.Bar(
            x=meses_list,
            y=profit_usd_list,
            name="Beneficio Mensual ($)",
            marker_color=colores_barras,
            text=textos_fuera_barra,
            textposition='outside',
            textfont=dict(color='white', size=11),
            cliponaxis=False,
            hovertext=hover_barras,
            hovertemplate="%{hovertext}<extra></extra>"
        ),
        secondary_y=False,
    )

    fig_año.add_trace(
        go.Scatter(
            x=meses_list,
            y=dd_pct_list,
            name="Máx Drawdown del Mes (%)",
            mode='lines+markers',
            line=dict(color='orange', width=2),
            marker=dict(size=8, color='white', line=dict(width=2, color='orange')),
            hovertext=hover_lineas,
            hovertemplate="%{hovertext}<extra></extra>"
        ),
        secondary_y=True,
    )

    # ← NUEVO: Línea de media de retorno mensual (eje izquierdo)
    fig_año.add_trace(
        go.Scatter(
            x=meses_list,
            y=[media_profit_usd] * len(meses_list),
            name=f"Media Retorno Mensual ({media_profit_pct:+.2f}%)",
            mode='lines',
            line=dict(color='rgba(0, 200, 100, 0.5)', width=2, dash='dash'),
            hovertemplate=f"Media Retorno: ${media_profit_usd:,.2f} ({media_profit_pct:+.2f}%)<extra></extra>"
        ),
        secondary_y=False,
    )

    # ← NUEVO: Línea de media de DD mensual (eje derecho)
    fig_año.add_trace(
        go.Scatter(
            x=meses_list,
            y=[media_dd_pct] * len(meses_list),
            name=f"Media DD Mensual ({media_dd_pct:.2f}%)",
            mode='lines',
            line=dict(color='rgba(255, 165, 0, 0.5)', width=2, dash='dash'),
            hovertemplate=f"Media DD: {media_dd_pct:.2f}%<extra></extra>"
        ),
        secondary_y=True,
    )

    profit_total_año_usd = sum(profit_usd_list)
    peor_dd_año_pct = min(dd_pct_list) if dd_pct_list else 0
    peor_dd_año_usd = min(dd_usd_list) if dd_usd_list else 0

    fig_año.update_layout(
        title=f'<b>Desempeño Mensual vs Capital Inicial ($10K) - {año}</b><br><sup>Profit Anual: ${profit_total_año_usd:,.2f} | Peor Caída (sobre 10K): {peor_dd_año_pct:.2f}% (${peor_dd_año_usd:,.2f})</sup>',
        template='plotly_dark',
        hovermode="x unified",
        width=950,
        height=500,                                      # ← +50px para dar aire
        showlegend=True,
        legend=dict(
            orientation="h",
            yanchor="top",                               # ← antes "bottom"
            y=-0.15,                                     # ← leyenda debajo del gráfico
            xanchor="center",                            # ← centrada
            x=0.5
        ),
        margin=dict(t=80, b=80)                          # ← margen top y bottom
    )

    fig_año.update_yaxes(title_text="<b>Beneficio ($)</b>", secondary_y=False, showgrid=False)
    fig_año.update_yaxes(title_text="<b>Drawdown (%)</b>", secondary_y=True, showgrid=True, gridcolor='rgba(255,255,255,0.1)')

    fig_año.show()
    fig_año.write_html(f"desempeno_{año}.html", auto_open=False)